## Reddit Data Collection + VADER Labeling

This notebook collects Reddit comments from selected subreddits and assigns weak sentiment labels using the VADER lexicon-based method.  
The purpose of this stage is to construct a clean three-class dataset that can be used in later supervised experiments for the happiness index project.

### 0. Environment Setup and Dependency Check

To ensure reproducibility and avoid runtime errors caused by missing dependencies, all required Python packages are checked and installed automatically at the beginning of the notebook.

This step guarantees that the experimental environment is correctly configured before proceeding with data collection and sentiment analysis.

In [11]:
import importlib
import subprocess
import sys

def install_if_missing(package):
    try:
        importlib.import_module(package)
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"⬇️ Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

packages = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "nltk",
    "requests"
]

for pkg in packages:
    install_if_missing(pkg)

✅ numpy already installed
✅ pandas already installed
✅ matplotlib already installed
✅ seaborn already installed
✅ nltk already installed
✅ requests already installed


### 1. Clear proxy settings

This section removes proxy-related environment variables before making API requests.  
It helps reduce connection issues caused by inherited local network settings, especially when collecting data from external endpoints.

In [2]:
import os
for k in [
    'HTTP_PROXY','HTTPS_PROXY','http_proxy','https_proxy',
    'ALL_PROXY','all_proxy','NO_PROXY','no_proxy'
]:
    os.environ.pop(k, None)
print('Proxy cleared')

Proxy cleared


### 2. Build a requests session

A reusable HTTP session is created here with retry logic for temporary network errors.  
This makes the data collection process more stable and helps avoid failures caused by short-lived connection problems or rate-limit responses.

In [3]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def build_session():
    session = requests.Session()
    session.trust_env = False
    session.proxies = {}

    retry = Retry(total=3, backoff_factor=1,
                  status_forcelist=[429,500,502,503,504])
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

session = build_session()
print('Session ready')

Session ready


### 3. Load VADER sentiment analyser

In this section, the VADER sentiment tool is loaded from NLTK.  
VADER is suitable for short informal social media texts and is used here to generate weak labels for negative, neutral, and positive sentiment.

In [4]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.data.path.append('./nltk_data')
sia = SentimentIntensityAnalyzer()
print('VADER loaded')

VADER loaded


### 4. Define data collection settings

This section sets the main parameters for the Reddit collection process, including target subreddits, date range, page size, and sample size per subreddit.  
Keeping these settings explicit makes the experiment easier to reproduce and adjust later.

In [5]:
import pandas as pd
from datetime import datetime, timezone
import time

SUBREDDITS = ['MadeMeSmile','depression','AskReddit']
START_DATE = '2022-01-01'
END_DATE = '2022-12-31'
TARGET_PER_SUBREDDIT = 10000
PAGE_SIZE = 500
API = 'https://api.pullpush.io/reddit/search/comment/'

### 5. Convert dates to Unix timestamps

The Reddit API uses Unix timestamps for time-based filtering.  
This helper function converts readable calendar dates into the timestamp format needed for the collection queries.

In [6]:
def to_unix(date_str):
    dt = datetime.strptime(date_str, '%Y-%m-%d').replace(tzinfo=timezone.utc)
    return int(dt.timestamp())

### 6. Filter invalid comments

Before labelling, raw comments need to be filtered to remove deleted, removed, empty, or uninformative texts.  
This step improves data quality and reduces noise in the final weakly labelled dataset.

In [7]:
def is_valid(text):
    if text in ['[deleted]','[removed]']:
        return False
    if text is None or len(text.strip()) < 5:
        return False
    if text.startswith('http'):
        return False
    return True

### 7. Apply VADER sentiment labels

This section defines the rule used to convert VADER compound scores into three sentiment classes.  
Following common VADER thresholds, comments are assigned to negative, neutral, or positive categories together with their original sentiment score.

In [8]:
def vader_label(text):
    score = sia.polarity_scores(text)['compound']
    if score > 0.05:
        return 2, score
    elif score < -0.05:
        return 0, score
    else:
        return 1, score

### 8. Fetch Reddit comments and label them

The main collection function requests Reddit comments in batches, applies text filtering, generates VADER labels, and stores the results in a structured format.  
This function is the core of the dataset-building pipeline.

In [9]:
def fetch(subreddit, start_ts, end_ts, target_n):
    results = []
    after = start_ts

    while len(results) < target_n:
        params = {
            'subreddit': subreddit,
            'after': after,
            'before': end_ts,
            'size': PAGE_SIZE,
            'sort': 'asc'
        }

        try:
            r = session.get(API, params=params, timeout=30)
            payload = r.json()

            if 'data' not in payload:
                print('No data field:', payload)
                break

            data = payload['data']

            if not data:
                break

            for item in data:
                text = item.get('body')
                if not is_valid(text):
                    continue

                label, score = vader_label(text)

                results.append({
                    'text': text,
                    'label': label,
                    'score': score,
                    'subreddit': subreddit,
                    'created_utc': item['created_utc']
                })

                if len(results) >= target_n:
                    break

            after = data[-1]['created_utc'] + 1
            print(f'{subreddit}: {len(results)} collected')
            time.sleep(1)

        except Exception as e:
            print(f'Error in {subreddit}:', e)
            break

    return results

### 9. Run collection and preview the dataset

Finally, the collection pipeline is executed for each selected subreddit.  
The results are merged into a single DataFrame, which can then be inspected and saved for later modelling stages.

In [10]:
start_ts = to_unix(START_DATE)
end_ts = to_unix(END_DATE)

all_data = []

for sr in SUBREDDITS:
    print(f'Fetching {sr}')
    data = fetch(sr, start_ts, end_ts, TARGET_PER_SUBREDDIT)
    all_data.extend(data)

df = pd.DataFrame(all_data)
print(df.shape)
df.head()

Fetching MadeMeSmile
MadeMeSmile: 94 collected
MadeMeSmile: 189 collected
MadeMeSmile: 278 collected
MadeMeSmile: 369 collected
MadeMeSmile: 458 collected
MadeMeSmile: 543 collected
MadeMeSmile: 622 collected
MadeMeSmile: 709 collected
MadeMeSmile: 794 collected
MadeMeSmile: 888 collected
MadeMeSmile: 977 collected
MadeMeSmile: 1050 collected
MadeMeSmile: 1143 collected
MadeMeSmile: 1236 collected
MadeMeSmile: 1324 collected
MadeMeSmile: 1414 collected
MadeMeSmile: 1505 collected
MadeMeSmile: 1600 collected
MadeMeSmile: 1696 collected
MadeMeSmile: 1789 collected
MadeMeSmile: 1883 collected
MadeMeSmile: 1962 collected
MadeMeSmile: 2041 collected
MadeMeSmile: 2128 collected
MadeMeSmile: 2214 collected
MadeMeSmile: 2311 collected
MadeMeSmile: 2401 collected
MadeMeSmile: 2475 collected
MadeMeSmile: 2554 collected
MadeMeSmile: 2641 collected
MadeMeSmile: 2727 collected
MadeMeSmile: 2813 collected
MadeMeSmile: 2893 collected
MadeMeSmile: 2984 collected
MadeMeSmile: 3078 collected
MadeMeSmile

,text,label,score,subreddit,created_utc
0,![gif](giphy|mFYTaY7Gth86xnE6N5),1,0.0000,MadeMeSmile,1640995203
1,Dayvon the boulder johnson.,1,0.0000,MadeMeSmile,1640995209
2,Congrats!! That's awesome!!,2,0.8647,MadeMeSmile,1640995213
3,"As someone who has never been homeless, I can ...",0,-0.5423,MadeMeSmile,1640995213
4,Congrats u f’ing legend!!!,2,0.6458,MadeMeSmile,1640995214
